# ARM97 Pressure-Level Profile Plotting

This notebook focuses on ARM97 baseline history variables with a vertical level dimension. It converts model hybrid levels to pressure levels, then plots time series at a selected pressure level.

Behavior:

- Variables with explicit ARM97 observation mappings: plot `observation + baseline`; lower panel is `model - observation`.
- Variables without mapped observations: plot `baseline`; lower panel is `sample - baseline` when experiment samples are enabled.
- Optional experiment members can be added in batches, for example 70 QMC samples as batches of 10.
- Numeric variables with dimensions like `time, lev, ...` or `time, ilev, ...` are discovered automatically; non-time/non-level variables are ignored.


In [ ]:
from __future__ import annotations

from dataclasses import dataclass
from datetime import timedelta
import os
from pathlib import Path
import re


def find_repo_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for candidate in (start, *start.parents):
        if (candidate / "e3sm_scm_run_scripts_baseline").exists():
            return candidate
    return Path("/Users/yunlong/Workshop/SCM-UQ-Workflow")


ROOT = Path(os.environ.get("SCM_UQ_WORKFLOW_ROOT", find_repo_root())).resolve()
os.environ.setdefault("MPLCONFIGDIR", str(ROOT / ".local_cache/matplotlib-cache"))
os.environ.setdefault("XDG_CACHE_HOME", str(ROOT / ".local_cache"))

import numpy as np
import pandas as pd
from netCDF4 import Dataset, num2date

print("repo root:", ROOT)


## Configuration

`TARGET_LEVEL_PA` controls the default pressure level for static export and the initial interactive view. Use `INCLUDE_LEVEL_VARIABLES` to limit the notebook while iterating; leave it as `None` to discover all level variables.

Set `INCLUDE_EXPERIMENT_MEMBERS = True` to add sample/member files in batches. The defaults point to the current 70-member qmc14x5 stitched experiment.


In [ ]:
DEFAULT_BASELINE = (
    ROOT
    / "e3sm_scm_run_scripts_baseline/baseline-output/scm_ARM97_baseline/run"
    / "case_scripts.eam.h0.1997-06-19-84585.nc"
)
DEFAULT_OBSERVATION = ROOT / "e3sm_scm_run_scripts_baseline/ARM97_iopfile_4scam.nc"
DEFAULT_EXPERIMENT_DIR = (
    ROOT
    / "arm97_experiments_0602/arm97_qmc14x5_stitched_seed20260602/mac/stitched"
)

BASELINE = Path(os.environ.get("SCM_BASELINE_HISTORY_FILE", DEFAULT_BASELINE)).expanduser().resolve()
OBSERVATION = Path(os.environ.get("ARM97_IOP_FILE", DEFAULT_OBSERVATION)).expanduser().resolve()
OUT_DIR = Path(os.environ.get("ARM97_PROFILE_PLOT_OUT_DIR", ROOT / "baseline_arm97_comparison/profile_level_figures")).expanduser().resolve()

TARGET_LEVEL_PA = 50000.0
MAX_PRESSURE_PA = 96500.0
INCLUDE_LEVEL_VARIABLES: list[str] | None = None
EXCLUDE_LEVEL_VARIABLES = set()

INCLUDE_EXPERIMENT_MEMBERS = True
EXPERIMENT_DIR = Path(os.environ.get("SCM_EXPERIMENT_HISTORY_DIR", DEFAULT_EXPERIMENT_DIR)).expanduser().resolve()
EXPERIMENT_FILE_GLOB = os.environ.get("SCM_EXPERIMENT_FILE_GLOB", "mac_ARM97_qmc14x5_*_stitched_26day.nc")
EXPERIMENT_RUN_ID_REGEX = os.environ.get("SCM_EXPERIMENT_RUN_ID_REGEX", r"_(\d{3})_stitched_26day\.nc$")
EXPERIMENT_LABEL = os.environ.get("SCM_EXPERIMENT_CASE_LABEL", EXPERIMENT_DIR.parent.parent.name)
BATCH_SIZE = 10
BATCH_INDEX = 0
SELECTED_RUN_IDS: list[int | str] | None = None

assert BASELINE.exists(), BASELINE
assert OBSERVATION.exists(), OBSERVATION
if INCLUDE_EXPERIMENT_MEMBERS:
    assert EXPERIMENT_DIR.exists(), EXPERIMENT_DIR

print("baseline:", BASELINE)
print("observation:", OBSERVATION)
print("figure output:", OUT_DIR)
print("include experiment members:", INCLUDE_EXPERIMENT_MEMBERS)
if INCLUDE_EXPERIMENT_MEMBERS:
    print("experiment dir:", EXPERIMENT_DIR)
    print("experiment glob:", EXPERIMENT_FILE_GLOB)


## Observation Mapping

Only clear ARM97 pressure-level correspondences are mapped by default. Add entries here if you verify more model-to-observation relationships.


In [ ]:
@dataclass(frozen=True)
class LevelSpec:
    model: str
    obs: str | None
    units: str
    description: str = ""
    scale_obs: float = 1.0
    obs_offset: float = 0.0


OBSERVED_LEVEL_SPECS = {
    "T": LevelSpec("T", "T", "K", "temperature"),
    "Q": LevelSpec("Q", "q", "kg/kg", "specific humidity"),
    "U": LevelSpec("U", "u", "m/s", "zonal wind"),
    "V": LevelSpec("V", "v", "m/s", "meridional wind"),
    "OMEGA": LevelSpec("OMEGA", "omega", "Pa/s", "pressure vertical velocity"),
    "RELHUM": LevelSpec("RELHUM", "rh", "%", "relative humidity"),
}


## Helpers


In [ ]:
def filled(arr):
    return np.asarray(np.ma.asarray(arr, dtype=np.float64).filled(np.nan), dtype=np.float64)


def as_time_series(var):
    data = np.ma.asarray(var[:], dtype=np.float64)
    if data.ndim == 1:
        return filled(data)
    axes = tuple(range(1, data.ndim))
    return filled(np.ma.mean(data, axis=axes))


def load_time_axis(ds):
    t = ds.variables["time"]
    days = np.asarray(t[:], dtype=np.float64)
    dates = np.array(num2date(days, t.units, getattr(t, "calendar", "standard"), only_use_cftime_datetimes=False))
    return days, dates


def interpolate_time_series(source_days, source_values, target_days):
    source_values = np.asarray(source_values, dtype=np.float64)
    finite = np.isfinite(source_values)
    if finite.sum() < 2:
        return np.full_like(target_days, np.nan, dtype=np.float64)
    return np.interp(target_days, source_days[finite], source_values[finite], left=np.nan, right=np.nan)


def stats(model_values, obs_values):
    finite = np.isfinite(model_values) & np.isfinite(obs_values)
    if not finite.any():
        return {"n": 0, "mean_obs": np.nan, "mean_baseline": np.nan, "bias": np.nan, "mae": np.nan, "rmse": np.nan, "max_abs": np.nan}
    diff = model_values[finite] - obs_values[finite]
    return {
        "n": int(diff.size),
        "mean_obs": float(np.mean(obs_values[finite])),
        "mean_baseline": float(np.mean(model_values[finite])),
        "bias": float(np.mean(diff)),
        "mae": float(np.mean(np.abs(diff))),
        "rmse": float(np.sqrt(np.mean(diff * diff))),
        "max_abs": float(np.max(np.abs(diff))),
    }


def baseline_only_stats(values):
    finite = np.isfinite(values)
    if not finite.any():
        return {"n": 0, "mean_baseline": np.nan, "min_baseline": np.nan, "max_baseline": np.nan}
    x = values[finite]
    return {"n": int(x.size), "mean_baseline": float(np.mean(x)), "min_baseline": float(np.min(x)), "max_baseline": float(np.max(x))}


def is_numeric_time_level_var(var) -> bool:
    dims = getattr(var, "dimensions", ())
    if not dims or dims[0] != "time":
        return False
    if "lev" not in dims and "ilev" not in dims:
        return False
    try:
        return np.issubdtype(np.dtype(var.dtype), np.number)
    except TypeError:
        return False


def vertical_dim_for(var) -> str:
    dims = getattr(var, "dimensions", ())
    if "lev" in dims:
        return "lev"
    if "ilev" in dims:
        return "ilev"
    raise ValueError(f"No lev/ilev dimension in {getattr(var, 'name', var)}")


def time_level_matrix(var, vertical_dim: str) -> np.ndarray:
    dims = list(var.dimensions)
    data = np.ma.asarray(var[:], dtype=np.float64)
    time_axis = dims.index("time")
    level_axis = dims.index(vertical_dim)
    data = np.moveaxis(data, [time_axis, level_axis], [0, 1])
    if data.ndim > 2:
        data = np.ma.mean(data, axis=tuple(range(2, data.ndim)))
    return filled(data)


def model_pressure_matrix(ds, vertical_dim: str) -> np.ndarray:
    p0 = float(np.asarray(ds.variables["P0"][...]))
    if vertical_dim == "lev":
        hya = np.asarray(ds.variables["hyam"][:], dtype=np.float64)
        hyb = np.asarray(ds.variables["hybm"][:], dtype=np.float64)
    elif vertical_dim == "ilev":
        hya = np.asarray(ds.variables["hyai"][:], dtype=np.float64)
        hyb = np.asarray(ds.variables["hybi"][:], dtype=np.float64)
    else:
        raise ValueError(vertical_dim)
    ps = as_time_series(ds.variables["PS"])
    return hya[None, :] * p0 + hyb[None, :] * ps[:, None]


def interp_matrix_to_pressure(values: np.ndarray, pressure: np.ndarray, target_pressure: float) -> np.ndarray:
    values = np.asarray(values, dtype=np.float64)
    pressure = np.asarray(pressure, dtype=np.float64)
    out = np.full(values.shape[0], np.nan, dtype=np.float64)
    for i in range(values.shape[0]):
        p = pressure[i]
        v = values[i]
        finite = np.isfinite(p) & np.isfinite(v)
        if finite.sum() < 2:
            continue
        p = p[finite]
        v = v[finite]
        order = np.argsort(p)
        p = p[order]
        v = v[order]
        unique_p, unique_idx = np.unique(p, return_index=True)
        if unique_p.size < 2 or target_pressure < unique_p[0] or target_pressure > unique_p[-1]:
            continue
        out[i] = np.interp(target_pressure, unique_p, v[unique_idx])
    return out


def obs_expression_available(obs, expression: str | None) -> bool:
    return expression is not None and expression in obs.variables


def obs_time_level_matrix(var) -> np.ndarray:
    dims = list(var.dimensions)
    data = np.ma.asarray(var[:], dtype=np.float64)
    time_axis = dims.index("time")
    level_axis = dims.index("lev")
    data = np.moveaxis(data, [time_axis, level_axis], [0, 1])
    if data.ndim > 2:
        data = np.ma.mean(data, axis=tuple(range(2, data.ndim)))
    return filled(data)


def parse_run_id(path: Path, regex: str = EXPERIMENT_RUN_ID_REGEX):
    match = re.search(regex, path.name)
    if not match:
        return None
    value = match.group(1)
    try:
        return int(value)
    except ValueError:
        return value


def sort_key(value):
    if isinstance(value, int):
        return (0, value)
    return (1, str(value))


def run_label(run_id) -> str:
    return f"{run_id:03d}" if isinstance(run_id, int) else str(run_id)


def discover_experiment_files() -> pd.DataFrame:
    rows = []
    for index, path in enumerate(sorted(EXPERIMENT_DIR.glob(EXPERIMENT_FILE_GLOB))):
        run_id = parse_run_id(path)
        rows.append({"run_id": index if run_id is None else run_id, "path": path})
    if not rows:
        raise FileNotFoundError(f"No files matched {EXPERIMENT_FILE_GLOB!r} in {EXPERIMENT_DIR}")
    return pd.DataFrame(rows).sort_values("run_id", key=lambda s: s.map(sort_key)).reset_index(drop=True)


def select_batch(files: pd.DataFrame, batch_size: int, batch_index: int, selected_run_ids=None) -> pd.DataFrame:
    if selected_run_ids is not None:
        selected = files[files["run_id"].isin(selected_run_ids)].copy()
        missing = sorted(set(selected_run_ids) - set(selected["run_id"]), key=sort_key)
        if missing:
            raise ValueError(f"Missing requested run ids: {missing}")
        return selected.sort_values("run_id", key=lambda s: s.map(sort_key)).reset_index(drop=True)
    if batch_size <= 0:
        raise ValueError("BATCH_SIZE must be positive")
    start = batch_index * batch_size
    stop = start + batch_size
    selected = files.iloc[start:stop].copy()
    if selected.empty:
        n_batches = int(np.ceil(len(files) / batch_size))
        raise ValueError(f"BATCH_INDEX {batch_index} is empty; valid range is 0..{n_batches - 1}")
    return selected.reset_index(drop=True)


## Discover And Load Level Variables

If `INCLUDE_EXPERIMENT_MEMBERS` is enabled, this cell also discovers the selected sample/member batch and interpolates those members to the same pressure levels as the baseline.


In [ ]:
def discover_level_specs() -> list[LevelSpec]:
    include = set(INCLUDE_LEVEL_VARIABLES) if INCLUDE_LEVEL_VARIABLES is not None else None
    specs = []
    with Dataset(BASELINE) as baseline, Dataset(OBSERVATION) as obs:
        for name, var in baseline.variables.items():
            if name in EXCLUDE_LEVEL_VARIABLES:
                continue
            if include is not None and name not in include:
                continue
            if not is_numeric_time_level_var(var):
                continue
            if name in OBSERVED_LEVEL_SPECS and obs_expression_available(obs, OBSERVED_LEVEL_SPECS[name].obs):
                specs.append(OBSERVED_LEVEL_SPECS[name])
            else:
                units = getattr(var, "units", "") or ""
                description = getattr(var, "long_name", "") or name
                specs.append(LevelSpec(name, None, units, description))
    if not specs:
        raise ValueError("No level variables discovered")
    return specs


def selected_experiment_batch():
    if not INCLUDE_EXPERIMENT_MEMBERS:
        return None, None
    all_files = discover_experiment_files()
    batch_files = select_batch(all_files, BATCH_SIZE, BATCH_INDEX, SELECTED_RUN_IDS)
    return all_files, batch_files


def load_member_level_values(member_file: Path, spec: LevelSpec, target_pressures_pa: np.ndarray):
    with Dataset(member_file) as member_ds:
        if spec.model not in member_ds.variables:
            return None
        var = member_ds.variables[spec.model]
        if not is_numeric_time_level_var(var):
            return None
        vertical_dim = vertical_dim_for(var)
        member_days, member_dates = load_time_axis(member_ds)
        member_matrix = time_level_matrix(var, vertical_dim)
        member_pressure = model_pressure_matrix(member_ds, vertical_dim)
        values_by_level = {
            float(level_pa): interp_matrix_to_pressure(member_matrix, member_pressure, float(level_pa))
            for level_pa in target_pressures_pa
        }
        return {"days": member_days, "dates": member_dates, "values_by_level": values_by_level}


def load_level_data(specs=None):
    specs = discover_level_specs() if specs is None else list(specs)
    data = {}
    summary_rows = []
    all_files, batch_files = selected_experiment_batch()

    if batch_files is not None:
        n_batches = int(np.ceil(len(all_files) / BATCH_SIZE))
        print(f"experiment members enabled: {len(all_files)} files; batch {BATCH_INDEX + 1}/{n_batches}: {[run_label(x) for x in batch_files['run_id']]}")

    with Dataset(BASELINE) as baseline, Dataset(OBSERVATION) as obs:
        model_days, model_dates = load_time_axis(baseline)
        obs_days = (np.asarray(obs.variables["tsec"][:], dtype=np.float64) - float(obs.variables["tsec"][0])) / 86400.0
        origin = model_dates[0] - timedelta(days=float(model_days[0]))
        obs_dates = np.array([origin + timedelta(days=float(x)) for x in obs_days])
        obs_levels_pa = np.asarray(obs.variables["lev"][:], dtype=np.float64)
        pressure_options_pa = np.array([float(p) for p in obs_levels_pa if float(p) <= MAX_PRESSURE_PA], dtype=np.float64)

        pressure_cache = {}
        for spec in specs:
            if spec.model not in baseline.variables:
                continue
            var = baseline.variables[spec.model]
            vertical_dim = vertical_dim_for(var)
            if vertical_dim not in pressure_cache:
                pressure_cache[vertical_dim] = model_pressure_matrix(baseline, vertical_dim)
            model_matrix = time_level_matrix(var, vertical_dim)
            model_pressure = pressure_cache[vertical_dim]
            has_obs = obs_expression_available(obs, spec.obs)
            obs_matrix = None
            if has_obs:
                obs_matrix = obs_time_level_matrix(obs.variables[spec.obs]) * spec.scale_obs + spec.obs_offset

            member_sources = []
            if batch_files is not None:
                for row in batch_files.itertuples(index=False):
                    member_payload = load_member_level_values(Path(row.path), spec, pressure_options_pa)
                    if member_payload is None:
                        continue
                    member_payload.update({"run_id": row.run_id, "path": Path(row.path)})
                    member_sources.append(member_payload)

            level_data = {}
            for target_pressure in pressure_options_pa:
                model_at_level = interp_matrix_to_pressure(model_matrix, model_pressure, float(target_pressure))
                item = {"model_at_level": model_at_level, "members": []}
                row = {
                    "variable": spec.model,
                    "observation": spec.obs if has_obs else "",
                    "description": spec.description,
                    "vertical_dim": vertical_dim,
                    "level_pa": float(target_pressure),
                    "level_hpa": float(target_pressure / 100.0),
                    "units": spec.units,
                    "comparison": "observation" if has_obs else "baseline_only",
                }
                obs_at_model = None
                if has_obs:
                    level_index = int(np.argmin(np.abs(obs_levels_pa - target_pressure)))
                    obs_native = obs_matrix[:, level_index]
                    obs_at_model = interpolate_time_series(obs_days, obs_native, model_days)
                    diff = model_at_level - obs_at_model
                    item.update({"obs_native": obs_native, "obs_at_model": obs_at_model, "diff": diff, "stats": stats(model_at_level, obs_at_model)})
                    row.update(item["stats"])
                else:
                    item.update({"stats": baseline_only_stats(model_at_level)})
                    row.update(item["stats"])

                for member in member_sources:
                    member_values = member["values_by_level"][float(target_pressure)]
                    baseline_at_member = interpolate_time_series(model_days, model_at_level, member["days"])
                    member_item = {
                        "run_id": member["run_id"],
                        "path": member["path"],
                        "dates": member["dates"],
                        "days": member["days"],
                        "values": member_values,
                        "baseline_at_member": baseline_at_member,
                        "diff_baseline": member_values - baseline_at_member,
                    }
                    if has_obs:
                        obs_at_member = interpolate_time_series(obs_days, obs_native, member["days"])
                        member_item["obs_at_member"] = obs_at_member
                        member_item["diff_obs"] = member_values - obs_at_member
                    item["members"].append(member_item)

                level_data[float(target_pressure)] = item
                summary_rows.append(row)

            data[spec.model] = {
                "spec": spec,
                "has_obs": has_obs,
                "vertical_dim": vertical_dim,
                "model_dates": model_dates,
                "model_days": model_days,
                "obs_dates": obs_dates,
                "obs_days": obs_days,
                "obs_levels_pa": obs_levels_pa,
                "pressure_options_pa": pressure_options_pa,
                "level_data": level_data,
            }

    summary = pd.DataFrame(summary_rows)
    if not summary.empty:
        summary = summary.sort_values(["comparison", "variable", "level_pa"]).reset_index(drop=True)
    return data, summary


LEVEL_SPECS = discover_level_specs()
PROFILE_DATA, PROFILE_SUMMARY = load_level_data(LEVEL_SPECS)
with_obs = sum(d["has_obs"] for d in PROFILE_DATA.values())
pressure_options = next(iter(PROFILE_DATA.values()))["pressure_options_pa"]
member_count = len(next(iter(PROFILE_DATA.values()))["level_data"][float(pressure_options[0])]["members"])
print(f"discovered {len(LEVEL_SPECS)} level variables")
print(f"variables with observation: {with_obs}; baseline-only variables: {len(PROFILE_DATA) - with_obs}")
print(f"pressure options: {len(pressure_options)} levels, {pressure_options.min()/100:.0f}-{pressure_options.max()/100:.0f} hPa")
print(f"experiment members per variable/level: {member_count}")
PROFILE_SUMMARY.head(20)


## Interactive Pressure-Level Time Series

When experiment members are enabled, the selected batch is added to the top panel. The lower panel shows `baseline/member - observation` for observed variables and `member - baseline` for baseline-only variables.


In [ ]:
def nearest_level(levels_pa, target_pa: float) -> float:
    levels_pa = np.asarray(levels_pa, dtype=np.float64)
    return float(levels_pa[np.argmin(np.abs(levels_pa - target_pa))])


def member_run_range(members) -> str:
    if not members:
        return ""
    ids = [m["run_id"] for m in members]
    return f" members {run_label(min(ids, key=sort_key))}-{run_label(max(ids, key=sort_key))}"


def draw_level_timeseries(name: str, level_pa: float):
    import matplotlib.dates as mdates
    import matplotlib.pyplot as plt
    from IPython.display import display

    item = PROFILE_DATA[name]
    spec = item["spec"]
    level_pa = nearest_level(item["pressure_options_pa"], float(level_pa))
    d = item["level_data"][level_pa]
    members = d.get("members", [])

    if item["has_obs"]:
        s = d["stats"]
        fig, axes = plt.subplots(2, 1, figsize=(12, 7), sharex=True, gridspec_kw={"height_ratios": [2.1, 1.0], "hspace": 0.08})
        axes[0].plot(item["obs_dates"], d["obs_native"], color="0.25", lw=1.1, alpha=0.75, label="observation")
        axes[0].plot(item["model_dates"], d["model_at_level"], color="#1261A6", lw=2.0, label="baseline")
        for member in members:
            label = run_label(member["run_id"])
            axes[0].plot(member["dates"], member["values"], lw=0.9, alpha=0.50, label=f"sample {label}")
        axes[0].set_ylabel(f"{name} ({spec.units})" if spec.units else name)
        axes[0].legend(loc="best", frameon=False, ncols=2, fontsize=8)
        axes[0].grid(True, alpha=0.25)

        axes[1].plot(item["model_dates"], d["diff"], color="#C2410C", lw=1.4, label="baseline - observation")
        for member in members:
            axes[1].plot(member["dates"], member["diff_obs"], lw=0.75, alpha=0.42)
        axes[1].axhline(0, color="black", lw=0.8)
        axes[1].set_ylabel(f"model - obs ({spec.units})" if spec.units else "model - obs")
        axes[1].set_xlabel("Time")
        axes[1].grid(True, alpha=0.25)
        axes[1].xaxis.set_major_locator(mdates.DayLocator(interval=4))
        axes[1].xaxis.set_major_formatter(mdates.DateFormatter("%b %d"))
        title = (
            f"{name} at {level_pa / 100:.0f} hPa: observation, baseline{member_run_range(members)}\n"
            f"{spec.description} | obs={spec.obs} | bias={s['bias']:.3g} {spec.units}, "
            f"RMSE={s['rmse']:.3g}, MAE={s['mae']:.3g}, n={s['n']}"
        )
    elif members:
        fig, axes = plt.subplots(2, 1, figsize=(12, 7), sharex=True, gridspec_kw={"height_ratios": [2.1, 1.0], "hspace": 0.08})
        axes[0].plot(item["model_dates"], d["model_at_level"], color="#1261A6", lw=2.0, label="baseline")
        for member in members:
            label = run_label(member["run_id"])
            axes[0].plot(member["dates"], member["values"], lw=0.9, alpha=0.50, label=f"sample {label}")
        axes[0].set_ylabel(f"{name} ({spec.units})" if spec.units else name)
        axes[0].legend(loc="best", frameon=False, ncols=2, fontsize=8)
        axes[0].grid(True, alpha=0.25)
        for member in members:
            axes[1].plot(member["dates"], member["diff_baseline"], lw=0.75, alpha=0.46)
        axes[1].axhline(0, color="black", lw=0.8)
        axes[1].set_ylabel(f"sample - baseline ({spec.units})" if spec.units else "sample - baseline")
        axes[1].set_xlabel("Time")
        axes[1].grid(True, alpha=0.25)
        axes[1].xaxis.set_major_locator(mdates.DayLocator(interval=4))
        axes[1].xaxis.set_major_formatter(mdates.DateFormatter("%b %d"))
        s = d["stats"]
        title = (
            f"{name} at {level_pa / 100:.0f} hPa: baseline{member_run_range(members)}\n"
            f"{spec.description} | no mapped observation | baseline mean={s['mean_baseline']:.3g} {spec.units}, n={s['n']}"
        )
    else:
        fig, axes = plt.subplots(1, 1, figsize=(12, 4.8))
        axes = [axes]
        axes[0].plot(item["model_dates"], d["model_at_level"], color="#1261A6", lw=2.0, label="baseline")
        axes[0].set_ylabel(f"{name} ({spec.units})" if spec.units else name)
        axes[0].set_xlabel("Time")
        axes[0].legend(loc="best", frameon=False)
        axes[0].grid(True, alpha=0.25)
        axes[0].xaxis.set_major_locator(mdates.DayLocator(interval=4))
        axes[0].xaxis.set_major_formatter(mdates.DateFormatter("%b %d"))
        s = d["stats"]
        title = (
            f"{name} at {level_pa / 100:.0f} hPa: baseline only\n"
            f"{spec.description} | no mapped observation | mean={s['mean_baseline']:.3g} {spec.units}, n={s['n']}"
        )

    fig.suptitle(title, x=0.02, ha="left", y=0.98)
    fig.autofmt_xdate(rotation=0)
    fig.subplots_adjust(top=0.84, left=0.08, right=0.98, bottom=0.12)
    display(fig)
    plt.close(fig)


try:
    import ipywidgets as widgets
    from IPython.display import display

    profile_names = list(PROFILE_DATA)
    pressure_options = [(f"{p / 100:.0f} hPa", float(p)) for p in next(iter(PROFILE_DATA.values()))["pressure_options_pa"]]
    default_pressure = nearest_level([value for _, value in pressure_options], TARGET_LEVEL_PA)

    profile_var = widgets.Dropdown(
        options=[(f"{name}: {PROFILE_DATA[name]['spec'].description}", name) for name in profile_names],
        value="T" if "T" in profile_names else profile_names[0],
        description="variable",
        layout=widgets.Layout(width="520px"),
    )
    pressure_level = widgets.SelectionSlider(
        options=pressure_options,
        value=default_pressure,
        description="level",
        continuous_update=False,
        readout=True,
        layout=widgets.Layout(width="650px"),
        style={"description_width": "50px"},
    )
    out = widgets.interactive_output(draw_level_timeseries, {"name": profile_var, "level_pa": pressure_level})
    display(widgets.VBox([profile_var, pressure_level]), out)
except Exception as exc:
    print("ipywidgets controls are unavailable in this kernel; drawing default view instead.")
    print(repr(exc))
    default_name = "T" if "T" in PROFILE_DATA else next(iter(PROFILE_DATA))
    draw_level_timeseries(default_name, TARGET_LEVEL_PA)


## Static Export

This exports one PNG per variable at `TARGET_LEVEL_PA`, plus a CSV containing all loaded variable/level summary rows. When experiment members are enabled, the selected batch is included in the PNGs.


In [ ]:
import matplotlib.pyplot as plt


def safe_name(text: str) -> str:
    return re.sub(r"[^A-Za-z0-9_.-]+", "_", text)


def export_level_figures(data: dict[str, dict] = PROFILE_DATA, target_level_pa: float = TARGET_LEVEL_PA, out_dir: Path = OUT_DIR) -> list[Path]:
    level_label = f"{nearest_level(next(iter(data.values()))['pressure_options_pa'], target_level_pa) / 100:.0f}hPa"
    batch_label = "baseline_only" if not INCLUDE_EXPERIMENT_MEMBERS else ("custom" if SELECTED_RUN_IDS is not None else f"batch{BATCH_INDEX:02d}")
    fig_dir = out_dir / level_label / batch_label
    fig_dir.mkdir(parents=True, exist_ok=True)
    paths = []
    for name in data:
        item = data[name]
        spec = item["spec"]
        level_pa = nearest_level(item["pressure_options_pa"], target_level_pa)
        d = item["level_data"][level_pa]
        members = d.get("members", [])

        if item["has_obs"]:
            s = d["stats"]
            fig, axes = plt.subplots(2, 1, figsize=(12, 7), sharex=True, gridspec_kw={"height_ratios": [2.1, 1.0], "hspace": 0.08})
            axes[0].plot(item["obs_dates"], d["obs_native"], color="0.25", lw=1.1, alpha=0.75, label="observation")
            axes[0].plot(item["model_dates"], d["model_at_level"], color="#1261A6", lw=2.0, label="baseline")
            for member in members:
                axes[0].plot(member["dates"], member["values"], lw=0.9, alpha=0.50, label=f"sample {run_label(member['run_id'])}")
            axes[0].set_ylabel(f"{name} ({spec.units})" if spec.units else name)
            axes[0].legend(loc="best", frameon=False, ncols=2, fontsize=8)
            axes[0].grid(True, alpha=0.25)
            axes[1].plot(item["model_dates"], d["diff"], color="#C2410C", lw=1.4)
            for member in members:
                axes[1].plot(member["dates"], member["diff_obs"], lw=0.75, alpha=0.42)
            axes[1].axhline(0, color="black", lw=0.8)
            axes[1].set_ylabel(f"model - obs ({spec.units})" if spec.units else "model - obs")
            axes[1].set_xlabel("Time")
            axes[1].grid(True, alpha=0.25)
            title = (
                f"{name} at {level_pa / 100:.0f} hPa: observation, baseline{member_run_range(members)}\n"
                f"{spec.description} | obs={spec.obs} | baseline bias={s['bias']:.4g} {spec.units}, RMSE={s['rmse']:.4g}, n={s['n']}"
            )
        elif members:
            s = d["stats"]
            fig, axes = plt.subplots(2, 1, figsize=(12, 7), sharex=True, gridspec_kw={"height_ratios": [2.1, 1.0], "hspace": 0.08})
            axes[0].plot(item["model_dates"], d["model_at_level"], color="#1261A6", lw=2.0, label="baseline")
            for member in members:
                axes[0].plot(member["dates"], member["values"], lw=0.9, alpha=0.50, label=f"sample {run_label(member['run_id'])}")
            axes[0].set_ylabel(f"{name} ({spec.units})" if spec.units else name)
            axes[0].legend(loc="best", frameon=False, ncols=2, fontsize=8)
            axes[0].grid(True, alpha=0.25)
            for member in members:
                axes[1].plot(member["dates"], member["diff_baseline"], lw=0.75, alpha=0.46)
            axes[1].axhline(0, color="black", lw=0.8)
            axes[1].set_ylabel(f"sample - baseline ({spec.units})" if spec.units else "sample - baseline")
            axes[1].set_xlabel("Time")
            axes[1].grid(True, alpha=0.25)
            title = (
                f"{name} at {level_pa / 100:.0f} hPa: baseline{member_run_range(members)}\n"
                f"{spec.description} | no mapped observation | baseline mean={s['mean_baseline']:.4g} {spec.units}, n={s['n']}"
            )
        else:
            s = d["stats"]
            fig, ax = plt.subplots(1, 1, figsize=(12, 4.8))
            ax.plot(item["model_dates"], d["model_at_level"], color="#1261A6", lw=2.0, label="baseline")
            ax.set_ylabel(f"{name} ({spec.units})" if spec.units else name)
            ax.set_xlabel("Time")
            ax.legend(loc="best", frameon=False)
            ax.grid(True, alpha=0.25)
            title = (
                f"{name} at {level_pa / 100:.0f} hPa: baseline only\n"
                f"{spec.description} | no mapped observation | mean={s['mean_baseline']:.4g} {spec.units}, n={s['n']}"
            )
        fig.suptitle(title, x=0.02, ha="left")
        fig.autofmt_xdate(rotation=0)
        fig.subplots_adjust(top=0.84, left=0.08, right=0.98, bottom=0.12)
        path = fig_dir / f"{safe_name(name)}_{level_label}_profile_level.png"
        fig.savefig(path, dpi=160, bbox_inches="tight")
        plt.close(fig)
        paths.append(path)
    return paths


OUT_DIR.mkdir(parents=True, exist_ok=True)
summary_out = OUT_DIR / "pressure_level_profile_summary.csv"
PROFILE_SUMMARY.to_csv(summary_out, index=False)
paths = export_level_figures(PROFILE_DATA, TARGET_LEVEL_PA, OUT_DIR)
print(summary_out)
print(f"exported {len(paths)} figures to {paths[0].parent if paths else OUT_DIR}")
paths[:5]


## Optional: Export Multiple Pressure Levels

Uncomment and edit `levels_to_export` if you want figures at several pressure levels.


In [ ]:
# levels_to_export = [85000.0, 70000.0, 50000.0, 30000.0]
# for level_pa in levels_to_export:
#     export_level_figures(PROFILE_DATA, level_pa, OUT_DIR)
